In [1]:
import os, shutil

In [10]:
import os
import zipfile
import numpy as np
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import io

nazwa_zip = "dogvscat_small.zip"

if not os.path.exists(nazwa_zip):
    print(f"❌ BŁĄD: Nie widzę pliku '{nazwa_zip}' w tym folderze!")
    print(f"Sprawdź, czy plik ZIP leży dokładnie obok Twojego notatnika .ipynb")
else:
    print(f"✅ Znaleziono plik {nazwa_zip}. Uruchamiam wczytywanie zdjęć z archiwum...")
    
    zdjecia = []
    etykiety = []
    
    # 2. Otwieramy plik ZIP bez rozpakowywania go na dysk
    with zipfile.ZipFile(nazwa_zip, 'r') as z:
        # Przeglądamy listę wszystkich plików wewnątrz ZIP-a
        for file_info in z.infolist():
            nazwa_pliku = file_info.filename
            
            # Zabezpieczenie: interesują nas tylko zdjęcia, ignorujemy pliki systemowe Maca
            if nazwa_pliku.lower().endswith(('.jpg', '.jpeg')) and '__macosx' not in nazwa_pliku.lower():
                try:
                    # Wyciągamy samą nazwę pliku (bez ścieżki folderów ze środka ZIP)
                    czysta_nazwa = os.path.basename(nazwa_pliku)
                    
                    if czysta_nazwa: # Jeśli to nie jest pusty katalog
                        # Odczytujemy zdjęcie bezpośrednio z pamięci ZIP
                        with z.open(file_info) as f:
                            img_data = f.read()
                            # Zamieniamy strumień bajtów na obiekt obrazu (rozmiar 150x150 jak u Cholleta)
                            img = load_img(io.BytesIO(img_data), target_size=(150, 150))
                            img_array = img_to_array(img) / 255.0 # Normalizacja pikseli do 0-1
                            
                            zdjecia.append(img_array)
                            
                            # Etykietowanie na podstawie czystej nazwy pliku
                            if czysta_nazwa.lower().startswith('dog'):
                                etykiety.append(1) # Pies
                            else:
                                etykiety.append(0) # Kot
                except Exception as e:
                    print(f"Pominięto problematyczny plik {nazwa_pliku}: {e}")

    # 3. Zamiana na gotowe tablice NumPy
    if len(zdjecia) > 0:
        X = np.array(zdjecia)
        y = np.array(etykiety)
        print(f"✅ Sukces! Pomyślnie wczytano {len(X)} zdjęć bezpośrednio z pliku ZIP.")
        print(f"Kształt danych: {X.shape}")
    else:
        print("❌ BŁĄD: Otworzono plik ZIP, ale nie znaleziono w nim odpowiednich zdjęć .jpg.")


✅ Znaleziono plik dogvscat_small.zip. Uruchamiam wczytywanie zdjęć z archiwum...
✅ Sukces! Pomyślnie wczytano 4000 zdjęć bezpośrednio z pliku ZIP.
Kształt danych: (4000, 150, 150, 3)
